# 15. SFT v2 (v2.1): 원본 Qwen의 좋은 답만 골라 다시 학습

14 심판 결과, 06 문장형 LoRA는 F1은 높았지만 **정답률 .736 → .450**, **잘못 거절 .056 → .102**로 원본보다 나빴습니다.
06 학습 데이터에서 찾은 원인과 이번 수정:

| | 06 (이전) | 15 (이번) |
|---|---|---|
| 정답 | MS MARCO 문장형 답 1문장 (짧아서 정보 빠짐) | **원본 Qwen 답 3개 생성 → 14B 심판이 정답 2점 + 근거 충실한 답만** |
| 거절 | `missed`(정답 문단이 top5 밖) + MS MARCO 답 없음 질문 → 문서에 답이 있어도 거절로 학습 | **14B 심판이 "top5 문단에 답 없음"으로 본 경우만** 거절, 비율 8% |
| 답 없음 질문인데 문단에 답이 있음 | 거절 | 근거 충실한 답이 있으면 **답변으로 학습** |
| 모델 고르기 | 학습 loss | 14와 같은 심판 점수 (정답률·잘못 거절) |

- 시험 650·검증 300은 14와 같은 답변 파일과 같은 심판 캐시(`judge_eval/judgments.jsonl`)를 써서 바로 비교됩니다.
- 학습 질문에서 시험·검증·`valgen_set.jsonl` 질문은 뺍니다.
- 단계별 결과를 드라이브에 저장하므로 중간에 끊겨도 **모두 실행**하면 이어서 진행합니다.
- **먼저 `QUICK = True`로 5~10분 점검** → 에러가 없으면 `False`로 바꾸고 *런타임 → 세션 다시 시작 후 모두 실행*. A100 기준 약 1.5~2시간.

## 1. 설치 + 경로

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (14에서 확인한 조합)

In [ ]:
QUICK = True   # True: 질문 수십 개로 점검 (결과는 sft_v2/quick) / False: 본 실행

import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'                     # 08 시험 답변
JUDGE_DIR  = f'{SAVE_DIR}/judge_eval'              # 14 검증 답변 + 심판 캐시
OUT_DIR    = f'{SAVE_DIR}/sft_v2' + ('/quick' if QUICK else '')
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(REPORT_DIR, exist_ok=True)
print('QUICK =', QUICK, '→', OUT_DIR)

## 2. 설정 + 공통 함수

In [ ]:
import re, json, gzip, time, random, string, hashlib, gc, contextlib
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch, transformers, peft, sentence_transformers

SEED      = 42
K, TOP_N  = 100, 5
THRESH    = 4.921875                 # 08 챗봇과 같은 거절 기준 (MiniLM 최고점)
GEN_LLM   = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
EMB_MODEL = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
NO_ANSWER = 'No Answer Present.'

N_ANS_Q    = 40 if QUICK else 2000   # 학습 후보: 답 있는 train 질문
N_NOANS_Q  = 16 if QUICK else 600    # 학습 후보: MS MARCO 답 없음 질문
N_SAMPLES  = 3                       # 질문당 후보 답 (탐욕 1 + 샘플링 2)
TEMP       = 0.7
GEN_BS     = 16
CHUNK      = 256                     # 이만큼마다 드라이브에 저장 (끊겨도 이어서)
EVID_NO, EVID_YES = 0.2, 0.8         # 심판 P(문단에 답 있음) 기준: 거절 라벨 / 답 없음 질문인데 답 있음
REFUSE_FRAC = 0.08                   # 학습 예시 중 거절 비율 상한
MAX_LEN    = 1280
EPOCHS, LR = 1, 1e-4
BATCH, GRAD_ACC = 8, 2
LORA_R, LORA_ALPHA = 16, 32
N_TEST_Q   = 30 if QUICK else None   # 시험 650 중 일부만 (QUICK)
N_VAL_Q    = 12 if QUICK else None
DEMO = ['why does my face turn red and itchy after drinking alcohol',
        'what are the symptoms of low vitamin d',
        'can ibuprofen cause stomach bleeding',
        'how long does the flu last',
        'what is a normal resting heart rate for adults']

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')
print(f'torch {torch.__version__} / transformers {transformers.__version__} / peft {peft.__version__} / '
      f'sentence-transformers {sentence_transformers.__version__}')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path, mode='w'):
    with open(path, mode, encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f'GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]
docs_text_of = lambda r: '\n'.join(f'[{i}] ({d["site"]}) {texts[d["pid"]]}' for i, d in enumerate(r['rag_docs'], 1))
def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {texts[d["pid"]]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]
print(f'문단 {len(texts):,}')

## 3. 학습 후보 질문 + 검색 (08과 같은 설정)

- train 건강 질문 중 문단이 DB에 있는 것. 시험·검증·검증 생성셋(`valgen_set.jsonl`) 질문 제외
- 참고 정답 = MS MARCO 답 + 문장형 답 (심판이 비교)
- 검색 최고점 < 4.92면 챗봇이 LLM을 안 부르므로 학습에서도 제외

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
emb_model = SentenceTransformer(EMB_MODEL, device='cuda').half()
reranker = CrossEncoder(RERANKER, max_length=256, device='cuda'); reranker.model.half()
norm_text = lambda t: ' '.join(t.lower().split())

def retrieve_many(queries):
    qe = emb_model.encode([QUERY_PREFIX + q for q in queries], batch_size=128, normalize_embeddings=True,
                          convert_to_numpy=True).astype('float32')
    _, ids = index.search(qe, K)
    sc = reranker.predict([(q, texts[p]) for q, c in zip(queries, ids.tolist()) for p in c],
                          batch_size=512, show_progress_bar=len(queries) > 500).reshape(len(queries), K)
    out = []
    for cands, s_ in zip(ids.tolist(), sc):
        docs, seen = [], set()
        for i in np.argsort(-s_):
            p = int(cands[i]); key = norm_text(texts[p])
            if key in seen:
                continue
            seen.add(key); docs.append({'pid': p, 'score': float(s_[i]), 'site': sites[p]})
            if len(docs) == TOP_N:
                break
        out.append(docs)
    return out

eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
valgen   = load_jsonl(f'{JUDGE_DIR}/valgen_set.jsonl')
held_out = {str(q['qid']) for q in eval_set + val_set + valgen}
pid_of   = {t: i for i, t in enumerate(texts)}

POOL_PATH = f'{OUT_DIR}/pool.jsonl'
if os.path.exists(POOL_PATH):
    pool = load_jsonl(POOL_PATH)
    print('저장된 후보 질문 불러옴')
else:
    ans_pool, noans_pool = [], []
    for q in load_jsonl(f'{SAVE_DIR}/train_health_qa.jsonl.gz'):
        if not q['in_corpus'] or str(q['qid']) in held_out:
            continue
        gold = [pid_of[p['text']] for p in q['passages'] if p['is_selected'] == 1 and p['text'] in pid_of]
        if q['answers'] == [NO_ANSWER]:
            noans_pool.append({'qid': str(q['qid']), 'query': q['query'], 'refs': [], 'gold_pids': [], 'group': 'N'})
        else:
            refs = list(dict.fromkeys([a for a in q['answers'] + q['well_formed'] if a.strip()]))
            ans_pool.append({'qid': str(q['qid']), 'query': q['query'], 'refs': refs, 'gold_pids': gold, 'group': 'A'})
    rng = random.Random(15); rng.shuffle(ans_pool); rng.shuffle(noans_pool)
    pick = ans_pool[:int(N_ANS_Q * 1.15)] + noans_pool[:int(N_NOANS_Q * 1.15)]   # 검색 기준 탈락분 여유
    t = time.time()
    docs_all = retrieve_many([q['query'] for q in pick])
    pool = [{**q, 'rag_docs': d} for q, d in zip(pick, docs_all) if d[0]['score'] >= THRESH]
    pool = [q for q in pool if q['group'] == 'A'][:N_ANS_Q] + [q for q in pool if q['group'] == 'N'][:N_NOANS_Q]
    save_jsonl(pool, POOL_PATH)
    print(f'후보 풀: 답 있음 {len(ans_pool):,} / 답 없음 {len(noans_pool):,} → 검색 {len(pick):,}개 {time.time() - t:.0f}초')
for q in pool:
    q['gold_in_top5'] = any(d['pid'] in q['gold_pids'] for d in q['rag_docs'])
nA = sum(q['group'] == 'A' for q in pool)
print(f'학습 후보: 답 있음 {nA} (정답 문단 top5 안 {sum(q["gold_in_top5"] for q in pool)}) + 답 없음 {len(pool) - nA}')

## 4. 원본 Qwen으로 후보 답 생성 (질문당 3개: 탐욕 1 + 샘플링 2)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
CAND_PATH = f'{OUT_DIR}/candidates.jsonl'
done = {r['qid']: r for r in load_jsonl(CAND_PATH)} if os.path.exists(CAND_PATH) else {}
todo = [q for q in pool if q['qid'] not in done]
print(f'생성 완료 {len(done)} / 남음 {len(todo)}')

tok = AutoTokenizer.from_pretrained(GEN_LLM, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(GEN_LLM, dtype=torch.bfloat16, device_map='cuda').eval()

@torch.inference_mode()
def generate(model, msgs_list, n=1, sample=False, bs=GEN_BS, max_new=256):
    outs = []
    kw = dict(do_sample=True, temperature=TEMP, top_p=0.9, top_k=None) if sample else \
         dict(do_sample=False, temperature=None, top_p=None, top_k=None)
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id, **kw)
        dec = [o.strip() for o in tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        outs += [dec[j * n:(j + 1) * n] for j in range(len(prompts))]
    return outs

torch.manual_seed(SEED)
t = time.time()
for s in range(0, len(todo), CHUNK):
    part = todo[s:s + CHUNK]
    msgs = [rag_messages(q['query'], q['rag_docs']) for q in part]
    greedy = generate(llm, msgs)
    sampled = generate(llm, msgs, n=N_SAMPLES - 1, sample=True)
    rows = [{**q, 'samples': g + sm} for q, g, sm in zip(part, greedy, sampled)]
    save_jsonl(rows, CAND_PATH, mode='a')
    done.update({r['qid']: r for r in rows})
    print(f'  {s + len(part)}/{len(todo)} ({(time.time() - t) / 60:.0f}분)')
cands = [done[q['qid']] for q in pool]
print(f'후보 답 {sum(len(r["samples"]) for r in cands):,}개 / 탐욕 답이 거절인 질문 {np.mean([is_refusal(r["samples"][0]) for r in cands]):.2f}')

## 5. 데모 질문: 원본 Qwen 답 (나중에 SFT v2와 비교)

In [ ]:
demo_rows = [{'query': q, 'rag_docs': d} for q, d in zip(DEMO, retrieve_many(DEMO))]
demo_base = generate(llm, [rag_messages(r['query'], r['rag_docs']) for r in demo_rows])
for r, a in zip(demo_rows, demo_base):
    r['원본 Qwen'] = a[0] if r['rag_docs'][0]['score'] >= THRESH else REFUSAL
    print(f"Q: {r['query']} (검색 최고점 {r['rag_docs'][0]['score']:.2f})\n  원본: {r['원본 Qwen'][:200]}")
del llm; free_gpu()

## 6. 14B 심판으로 라벨 정하기

- **정답 점수** 0/1/2, **근거 충실도** 0/1: 14와 같은 프롬프트·같은 캐시
- **문단에 답 있음** 0/1 (새로 추가): 질문 + top5 문단만 보고 "이 문단들로 답할 수 있나"
- 라벨
  - 답 있는 질문: 후보 중 **정답 ≥ 1.5 이고 근거 ≥ 0.5**인 답 중 최고 → 답변 예시. 없으면, 문단에 답 있음 < 0.2일 때만 거절 예시, 나머지는 버림
  - 답 없음 질문: 문단에 답 있음 < 0.2 → 거절 예시. ≥ 0.8이고 근거 ≥ 0.8인 답이 있으면 → 답변 예시. 나머지는 버림

In [ ]:
jtok = AutoTokenizer.from_pretrained(JUDGE_LLM, padding_side='left')
judge = AutoModelForCausalLM.from_pretrained(JUDGE_LLM, dtype=torch.bfloat16, device_map='cuda').eval()
DIGITS = [jtok.encode(d, add_special_tokens=False)[0] for d in '012']

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')
SYS_EVID = ('You check whether the given documents contain the information needed to answer a health question. '
            'Reply with one digit only.\n'
            '1 = at least one document directly answers the question\n'
            '0 = the documents do not answer the question (related topic only, or missing information)')

def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]
def faith_msgs(docs_text, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{docs_text}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]
def evid_msgs(q, docs_text):
    return [{'role': 'system', 'content': SYS_EVID},
            {'role': 'user', 'content': f'Documents:\n{docs_text}\n\nQuestion: {q}\n\nDo the documents answer it (0 or 1)?'}]

@torch.inference_mode()
def digit_probs(msgs_list, n_digits, bs=8):
    prompts = [jtok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
    order = np.argsort([len(p) for p in prompts])
    out = [None] * len(prompts)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = jtok([prompts[i] for i in idx], return_tensors='pt', padding=True).to(judge.device)
        logits = judge(**enc).logits[:, -1, DIGITS[:n_digits]].float()
        for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
            out[i] = p
    return out

JUDGE_PATH = f'{JUDGE_DIR}/judgments.jsonl'    # 14와 같은 캐시 (같은 질문·답은 다시 채점 안 함)
cache = {j['key']: j for j in load_jsonl(JUDGE_PATH)} if os.path.exists(JUDGE_PATH) else {}
key_of = lambda kind, q, a: hashlib.md5(f'{kind}|{q}|{a}'.encode()).hexdigest()

def score(kind, items, chunk=2000):
    # kind 'c' (q, refs, ans) / 'f' (q, docs_text, ans) / 'e' (q, docs_text) → 0~2 또는 0~1 점수 목록
    keyed = [(key_of(kind, it[0], it[-1]), it) for it in items]
    need = list({k: it for k, it in keyed if k not in cache}.items())
    t = time.time()
    for s in range(0, len(need), chunk):
        part = need[s:s + chunk]
        if kind == 'c':
            ps = digit_probs([correct_msgs(*it) for _, it in part], 3); vals = [float(p[1] + 2 * p[2]) for p in ps]
        elif kind == 'f':
            ps = digit_probs([faith_msgs(it[1], it[2]) for _, it in part], 2); vals = [float(p[1]) for p in ps]
        else:
            ps = digit_probs([evid_msgs(it[0], it[1]) for _, it in part], 2); vals = [float(p[1]) for p in ps]
        for (k, _), v in zip(part, vals):
            cache[k] = {'key': k, 'v': v}
        save_jsonl(list(cache.values()), JUDGE_PATH)
        if len(need) > chunk:
            print(f'  {kind}: {s + len(part)}/{len(need)} ({(time.time() - t) / 60:.0f}분)')
    if need:
        print(f'심판 {kind}: {len(need)}개 새로 채점 ({time.time() - t:.0f}초)')
    return [cache[k]['v'] for k, _ in keyed]

In [ ]:
for r in cands:
    r['docs_text'] = docs_text_of(r)
    r['uniq'] = [a for a in dict.fromkeys(r['samples']) if not is_refusal(a)]
c_items = [(r['query'], r['refs'], a) for r in cands if r['group'] == 'A' for a in r['uniq']]
f_items = [(r['query'], r['docs_text'], a) for r in cands for a in r['uniq']]
c_vals = iter(score('c', c_items))
f_vals = iter(score('f', f_items))
e_vals = score('e', [(r['query'], r['docs_text']) for r in cands])

labeled = []
for r, ev in zip(cands, e_vals):
    sc = []
    for a in r['uniq']:
        c = next(c_vals) if r['group'] == 'A' else None
        sc.append((a, c, next(f_vals)))
    r['evid'] = ev
    if r['group'] == 'A':
        good = [(c + f, a) for a, c, f in sc if c >= 1.5 and f >= 0.5]
        if good:
            label, target = 'answer', max(good)[1]
        elif ev < EVID_NO:
            label, target = 'refuse', REFUSAL
        else:
            label, target = 'drop', None
    else:
        good = [(f, a) for a, _, f in sc if f >= 0.8]
        if ev < EVID_NO:
            label, target = 'refuse', REFUSAL
        elif ev >= EVID_YES and good:
            label, target = 'answer_noref', max(good)[1]
        else:
            label, target = 'drop', None
    labeled.append({'qid': r['qid'], 'query': r['query'], 'group': r['group'], 'rag_docs': r['rag_docs'],
                    'gold_in_top5': r['gold_in_top5'], 'evid': ev, 'label': label, 'target': target,
                    'greedy': r['samples'][0], 'scores': sc})
save_jsonl(labeled, f'{OUT_DIR}/labeled.jsonl')

lab = pd.DataFrame(labeled)
print('[라벨 분포 (행=질문 종류, 열=라벨)]'); display(pd.crosstab(lab['group'], lab['label'], margins=True))
missed = lab[(lab.group == 'A') & ~lab.gold_in_top5]
print(f'06에서 거절로 학습했던 경우(정답 문단 top5 밖) {len(missed)}개 → 이번 라벨: {missed.label.value_counts().to_dict()}')
print(f'MS MARCO 답 없음 {int((lab.group == "N").sum())}개 → 이번 라벨: {lab[lab.group == "N"].label.value_counts().to_dict()}')
print(f'심판 P(문단에 답 있음) 평균: 답 있음 {lab[lab.group == "A"].evid.mean():.2f} / 답 없음 {lab[lab.group == "N"].evid.mean():.2f}')

In [ ]:
answers  = [r for r in labeled if r['label'] in ('answer', 'answer_noref')]
refusals = [r for r in labeled if r['label'] == 'refuse']
n_ref = min(len(refusals), int(len(answers) * REFUSE_FRAC / (1 - REFUSE_FRAC)))
random.Random(SEED).shuffle(refusals)
train_ex = answers + refusals[:n_ref]
random.Random(SEED).shuffle(train_ex)
print(f'학습 예시 {len(train_ex)}개 = 답변 {len(answers)} + 거절 {n_ref} (거절 후보 {len(refusals)})')
print(f'답변 길이: 평균 {np.mean([len(r["target"].split()) for r in answers]):.0f}단어 (06 문장형 정답은 약 14단어)')
for r in [r for r in answers if not r['gold_in_top5']][:2] + answers[:2]:
    print(f"- Q: {r['query']}\n  → {r['target'][:220]}")

## 7. LoRA 학습 (06과 같은 구조, 정답·거절 라벨만 교체)

In [ ]:
del judge; free_gpu()
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

ADAPTER_DIR = f'{OUT_DIR}/adapter'
tok = AutoTokenizer.from_pretrained(GEN_LLM)
llm = AutoModelForCausalLM.from_pretrained(GEN_LLM, dtype=torch.bfloat16, device_map='cuda')
if os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'):
    llm = PeftModel.from_pretrained(llm, ADAPTER_DIR)
    print('저장된 SFT v2 LoRA 불러옴 (학습 건너뜀)')
else:
    llm.config.use_cache = False
    llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    llm.enable_input_require_grads()
    llm = get_peft_model(llm, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type='CAUSAL_LM',
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                                         'gate_proj', 'up_proj', 'down_proj']))
    llm.print_trainable_parameters()

    def to_features(ex):
        prompt = tok.apply_chat_template(rag_messages(ex['query'], ex['rag_docs']), tokenize=False, add_generation_prompt=True)
        p_ids = tok(prompt, add_special_tokens=False)['input_ids']
        a_ids = tok(ex['target'] + '<|im_end|>', add_special_tokens=False)['input_ids']
        return {'input_ids': p_ids + a_ids, 'labels': [-100] * len(p_ids) + a_ids}
    feats = [f for f in map(to_features, train_ex) if len(f['input_ids']) <= MAX_LEN]
    print(f'학습 예시 {len(feats):,}개 / 평균 길이 {np.mean([len(f["input_ids"]) for f in feats]):.0f} 토큰')

    def collate(batch):
        n = max(len(f['input_ids']) for f in batch)
        pad = lambda xs, v: [x + [v] * (n - len(x)) for x in xs]
        return {'input_ids': torch.tensor(pad([f['input_ids'] for f in batch], tok.pad_token_id)),
                'attention_mask': torch.tensor(pad([[1] * len(f['input_ids']) for f in batch], 0)),
                'labels': torch.tensor(pad([f['labels'] for f in batch], -100))}

    CKPT_DIR = f'{OUT_DIR}/ckpt'
    total_steps = -(-len(feats) // (BATCH * GRAD_ACC)) * EPOCHS
    args = TrainingArguments(
        output_dir=CKPT_DIR, num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine',
        warmup_steps=max(1, int(total_steps * 0.03)),
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, bf16=True,
        logging_steps=10, save_steps=50, save_total_limit=2, report_to='none',
        remove_unused_columns=False, seed=SEED)
    trainer = Trainer(model=llm, args=args, train_dataset=feats, data_collator=collate)
    has_ckpt = os.path.isdir(CKPT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT_DIR))
    t = time.time()
    trainer.train(resume_from_checkpoint=has_ckpt or None)
    print(f'학습 끝: {(time.time() - t) / 60:.0f}분')
    llm.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR)
    pd.DataFrame([l for l in trainer.state.log_history if 'loss' in l]).to_csv(f'{OUT_DIR}/train_log.csv', index=False)
    del trainer; free_gpu()
llm.eval(); llm.config.use_cache = True; tok.padding_side = 'left'

## 8. SFT v2 답변 생성: 시험 650 (08 문단) + 검증 300 (14 문단) + 데모

In [ ]:
evq = {str(q['qid']): q for q in eval_set}
test_rows = load_jsonl(f'{RAG_DIR}/eval_answers.jsonl')
for r in test_rows:
    r['answers'] = evq[str(r['qid'])]['answers']
    r['group'] = 'A' if r['domain'] == 'health' and r['answerable'] else ('N' if r['domain'] == 'health' else 'O')
val_rows = load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl')
if QUICK:
    test_rows = ([r for r in test_rows if r['group'] == 'A'][:N_TEST_Q - 10] + [r for r in test_rows if r['group'] == 'N'][:5]
                 + [r for r in test_rows if r['group'] == 'O'][:5])
    val_rows = [r for r in val_rows if r['group'] == 'A'][:N_VAL_Q - 4] + [r for r in val_rows if r['group'] == 'N'][:4]

SFT_ANS_PATH = f'{OUT_DIR}/sft_answers.jsonl'
if os.path.exists(SFT_ANS_PATH):
    saved = {(r['set'], str(r['qid'])): r['SFT v2'] for r in load_jsonl(SFT_ANS_PATH)}
    for name, rows in [('test', test_rows), ('val', val_rows)]:
        for r in rows:
            r['SFT v2'] = saved[(name, str(r['qid']))]
    print('저장된 SFT v2 답변 불러옴')
else:
    t = time.time()
    for name, rows in [('test', test_rows), ('val', val_rows)]:
        todo = [r for r in rows if r['rag_docs'][0]['score'] >= THRESH]
        outs = generate(llm, [rag_messages(r['query'], r['rag_docs']) for r in todo])
        for r in rows:
            r['SFT v2'] = REFUSAL
        for r, a in zip(todo, outs):
            r['SFT v2'] = a[0]
    save_jsonl([{'set': n, 'qid': str(r['qid']), 'SFT v2': r['SFT v2']} for n, rows in [('test', test_rows), ('val', val_rows)]
                for r in rows], SFT_ANS_PATH)
    print(f'생성 끝 {(time.time() - t) / 60:.0f}분')

demo_sft = generate(llm, [rag_messages(r['query'], r['rag_docs']) for r in demo_rows])
for r, a in zip(demo_rows, demo_sft):
    r['SFT v2'] = a[0] if r['rag_docs'][0]['score'] >= THRESH else REFUSAL
del llm; free_gpu()

## 9. 같은 심판으로 비교 (14와 같은 기준)

In [ ]:
judge = AutoModelForCausalLM.from_pretrained(JUDGE_LLM, dtype=torch.bfloat16, device_map='cuda').eval()

def evaluate(rows, systems):
    table = {}
    for name in systems:
        A = [r for r in rows if r['group'] == 'A']
        ansA = [r for r in A if not is_refusal(r[name])]
        cs = dict(zip([id(r) for r in ansA], score('c', [(r['query'], r['answers'], r[name]) for r in ansA])))
        answered = [r for r in rows if not is_refusal(r[name])]
        fs = score('f', [(r['query'], docs_text_of(r), r[name]) for r in answered])
        c_all = [cs.get(id(r), 0.0) for r in A]
        g = lambda k: [r for r in rows if r['group'] == k]
        table[name] = {
            '정답률 (심판 2점)':      np.mean([c >= 1.5 for c in c_all]),
            '정답+부분 (심판 ≥ 1점)': np.mean([c >= 0.5 for c in c_all]),
            '평균 점수 (0~2)':        np.mean(c_all),
            'F1 (참고)':              np.mean([max(f1(r[name], x) for x in r['answers']) for r in A]),
            '근거 충실도':            np.mean([f >= 0.5 for f in fs]) if fs else np.nan,
            '잘못 거절':              np.mean([is_refusal(r[name]) for r in g('A')]),
            '답 없음 거절률':         np.mean([is_refusal(r[name]) for r in g('N')]) if g('N') else np.nan,
            '분야 밖 거절률':         np.mean([is_refusal(r[name]) for r in g('O')]) if g('O') else np.nan,
        }
    return pd.DataFrame(table).T.round(3)

SYSTEMS = ['RAG 개선', 'RAG 개선 + LoRA', 'SFT v2']
test_tab = evaluate(test_rows, SYSTEMS); print(f'[시험 {len(test_rows)} (08 문단)]'); display(test_tab)
val_tab  = evaluate(val_rows, SYSTEMS);  print(f'[검증 {len(val_rows)}]'); display(val_tab)

# 질문별 정답 점수 차이 부트스트랩 (SFT v2 - 원본)
A = [r for r in test_rows if r['group'] == 'A']
cv = lambda r, n: 0.0 if is_refusal(r[n]) else cache[key_of('c', r['query'], r[n])]['v']
d = np.array([(cv(r, 'SFT v2') >= 1.5) - (cv(r, 'RAG 개선') >= 1.5) for r in A], dtype=float)
bs = np.random.default_rng(SEED).choice(d, (2000, len(d))).mean(1)
ci = (np.percentile(bs, 2.5), np.percentile(bs, 97.5))
print(f'시험 정답률 차이 (SFT v2 - 원본): {d.mean():+.3f} (95% CI {ci[0]:+.3f} ~ {ci[1]:+.3f})')

print('\n[데모 질문]')
for r in demo_rows:
    print(f"Q: {r['query']}\n  원본:   {r['원본 Qwen'][:220]}\n  SFT v2: {r['SFT v2'][:220]}")

## 10. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
lab_tab = pd.crosstab(lab['group'], lab['label'], margins=True)
demo_md = '\n'.join(f"- **{r['query']}**\n  - 원본: {r['원본 Qwen']}\n  - SFT v2: {r['SFT v2']}" for r in demo_rows)
summary = '\n'.join([
    f'# 15 SFT v2 (v2.1){" [QUICK 점검]" if QUICK else ""} ({stamp} KST)', '',
    f'- 학습 데이터: 원본 Qwen 답 {N_SAMPLES}개/질문 → 14B 심판으로 선택. 학습 예시 {len(train_ex)} = 답변 {len(answers)} + 거절 {n_ref}',
    f'- 거절 라벨은 심판 P(문단에 답 있음) < {EVID_NO}일 때만. 06에서 거절로 학습했던 경우(정답 문단 top5 밖) {len(missed)}개 → {missed.label.value_counts().to_dict()}',
    f'- LoRA r{LORA_R}, lr {LR}, {EPOCHS} epoch. 어댑터 `{ADAPTER_DIR.replace(SAVE_DIR, "health")}`',
    f'- 시험 정답률 차이 (SFT v2 - 원본): {d.mean():+.3f} (95% CI {ci[0]:+.3f} ~ {ci[1]:+.3f})', '',
    '## 라벨 분포 (A=답 있음, N=MS MARCO 답 없음)', '', lab_tab.to_markdown(), '',
    f'## 시험 {len(test_rows)} (08 문단)', '', test_tab.to_markdown(), '',
    f'## 검증 {len(val_rows)}', '', val_tab.to_markdown(), '',
    '## 데모', '', demo_md,
])
paths = [f'{OUT_DIR}/results_summary.md'] + ([] if QUICK else [f'{REPORT_DIR}/15_sft_v2_summary.md'])
for path in paths:
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
print(summary)

## 11. 드라이브 저장 마무리 + 런타임 종료 (QUICK 점검일 땐 런타임 유지)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if not QUICK:
    runtime.unassign()